# PyTorch Fundamentals: Zero to Confident
**A Professional Deep Dive from First Principles to Fashion-MNIST Production Classifier**

Welcome to PyTorch Fundamentals! This notebook is your comprehensive, step-by-step masterclass in PyTorch. Whether you are running on an ordinary CPU or an 8GB NVIDIA GPU (like an RTX 4060), this notebook automatically detects your hardware and guides you through core deep learning engineering practices.

---

## Pedagogical Glossary
Here is our master reference table of terms. Every technical word is defined in plain language when it is first encountered.

| Term | One-Sentence Plain Definition |
| :--- | :--- |
| **Tensor** | A multi-dimensional grid of numbers (like a list, a spreadsheet, or a 3D cube) stored in computer memory. |
| **Rank / Dimension (`ndim`)** | The number of directions or axes along which numbers are arranged in a tensor. |
| **Shape** | A tuple of integers stating exactly how many numbers exist along each axis of a tensor. |
| **Data Type (`dtype`)** | The binary format and precision used to store each number in memory (e.g., 32-bit float, 64-bit integer). |
| **Indexing** | Selecting specific individual numbers or slices from a tensor using numeric coordinates. |
| **Slicing** | Extracting a sub-window of elements along an axis using start, stop, and step bounds (`start:stop:step`). |
| **Broadcasting** | PyTorch's rule-based stretching of smaller tensors across dimensions of size 1 so arithmetic works without copying data. |
| **Stride** | The number of memory steps needed to jump to the next element along each dimension in physical RAM. |
| **View** | A new tensor metadata window looking at the exact same physical memory buffer with a different shape or stride. |
| **Reshape** | A method that returns a view if memory is contiguous, or creates a memory copy if it is not. |
| **Contiguity** | Storing tensor elements sequentially in memory in the exact order traversed by row-major indexing. |
| **Device** | The physical hardware processor and memory where a tensor resides (either system RAM `cpu` or graphics VRAM `cuda:0`). |
| **NumPy Interoperability** | The ability to share memory directly between PyTorch CPU tensors and NumPy ndarrays without overhead. |


## Environment Setup & Automatic Device Selection
Before doing any tensor math, let's verify our environment and establish automatic hardware device selection.

**What we are about to do and why:**
We import `torch`, inspect the runtime versions, and check whether an NVIDIA CUDA-enabled GPU is accessible. We define a global `device` object so all subsequent tensors and models can automatically run on the fastest available hardware.

In [1]:
# 1. Import core PyTorch library
import torch
# 2. Import numerical computing library for array conversions
import numpy as np
# 3. Import plotting library for visual diagnostic plots
import matplotlib.pyplot as plt

# 4. Check if CUDA (GPU support) is available on this system
use_cuda = torch.cuda.is_available()
# 5. Automatically select the GPU device if available, otherwise fallback to CPU
device = torch.device("cuda:0" if use_cuda else "cpu")

# 6. Print system configuration details
print(f"PyTorch Version: {torch.__version__}")
print(f"Device Selected: {device}")
if use_cuda:
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


PyTorch Version: 2.14.0+cu130
Device Selected: cuda:0
GPU Hardware: NVIDIA GeForce RTX 4060 Laptop GPU
Available VRAM: 8.00 GB


**What we just saw:**
PyTorch confirmed our execution device. If you have an NVIDIA GPU, `device` will point to `cuda:0` with its VRAM capacity reported; otherwise, it cleanly defaults to your system `cpu`.

---
# Step 1: Tensors — Creation, Shapes, Dtypes, Indexing, Broadcasting, Views, GPU & NumPy

Tensors are the fundamental data structure in PyTorch. Everything from input image pixels to layer weights and loss gradients is represented as a tensor.


### 1.1 Tensor Creation
**1. The idea in ONE plain sentence:**
A tensor is a multi-dimensional array of numbers stored in contiguous blocks of computer memory.

**2. Real-life analogy:**
Think of a tensor as an ice-cube tray where each individual pocket holds a single number. A 1D tray is a single row, a 2D tray is a flat rectangular tray, and a 3D tray is a stack of rectangular trays inside a freezer.

**3. Tiny example worked by hand:**
- A scalar (0D): just the number `7.0`.
- A 1D vector of length 3: `[1.0, 2.0, 3.0]`.
- A 2D matrix of shape (2, 3):
  $$\begin{bmatrix} 1.0 & 2.0 & 3.0 \\ 4.0 & 5.0 & 6.0 \end{bmatrix}$$
  It has 2 rows and 3 columns, making 6 numbers in total.

**4. The math with every symbol explained in words:**
A tensor $T \in \mathbb{R}^{d_1 \times d_2 \times \dots \times d_k}$ is a mathematical object indexed by $k$ coordinates $(i_1, i_2, \dots, i_k)$.
- $\mathbb{R}$: Real numbers (numbers with decimal points).
- $k$: The rank (number of dimensions or axes).
- $d_m$: The size (length) along axis $m$.
- Total elements $N = \prod_{m=1}^k d_m$ (the product of all dimension lengths).

**5. Code explained line by line:**
Let's create tensors from Python lists, and using built-in generators (`zeros`, `ones`, `rand`).


In [2]:
# 1. Create a 1D tensor directly from a Python list
t_1d = torch.tensor([1.0, 2.0, 3.0])
# 2. Create a 2D tensor (matrix) from nested Python lists
t_2d = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
# 3. Create a tensor filled with zeros of shape (2, 3)
t_zeros = torch.zeros(2, 3)
# 4. Create a tensor filled with ones of shape (2, 3)
t_ones = torch.ones(2, 3)
# 5. Create a tensor with uniformly distributed random numbers in [0, 1)
t_rand = torch.rand(2, 3)

# 6. Display tensor values and their exact shapes
print("t_1d shape:", t_1d.shape, "| elements:", t_1d)
print("t_2d shape:", t_2d.shape, "| elements:\n", t_2d)
print("t_zeros shape:", t_zeros.shape)
print("t_rand shape:", t_rand.shape)


t_1d shape: torch.Size([3]) | elements: tensor([1., 2., 3.])
t_2d shape: torch.Size([2, 3]) | elements:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
t_zeros shape: torch.Size([2, 3])
t_rand shape: torch.Size([2, 3])


**What we just saw:**
We created 1D and 2D tensors. Notice that `torch.tensor` takes data as an input list, while helper functions like `torch.zeros` take dimension sizes.

**6. One common beginner mistake:**
Passing numbers as separate arguments instead of a list into `torch.tensor`. Writing `torch.tensor(1, 2, 3)` causes a `TypeError` because PyTorch expects a single sequence or array, like `torch.tensor([1, 2, 3])`.

**7. Quick "check yourself" question:**
*Question:* What is the total number of elements in a tensor of shape `(3, 4, 5)`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> $3 \times 4 \times 5 = 60$ elements.
</details>

**Short Exercise 1.1:**
Create a 3D tensor of shape `(2, 3, 4)` initialized with all ones, and compute its total element count using `.numel()`.


In [3]:
# Exercise 1.1: Create 3D tensor and verify number of elements
# 1. Initialize a (2, 3, 4) tensor with ones
ex_ones = torch.ones(2, 3, 4)
# 2. Count total elements using the .numel() method
total_elements = ex_ones.numel()
# 3. Print the shape and total elements
print("Exercise 1.1 shape:", ex_ones.shape)
print("Exercise 1.1 total elements (numel):", total_elements)
assert total_elements == 24, "Expected 24 elements!"


Exercise 1.1 shape: torch.Size([2, 3, 4])
Exercise 1.1 total elements (numel): 24


### 1.2 Shapes and Dimensions: The NCHW Layout in Computer Vision
**1. The idea in ONE plain sentence:**
Shape tells you how many numbers are stacked along each direction, which in deep vision follows the standard order: Batch size, Channels, Height, and Width ($B \times C \times H \times W$).

**2. Real-life analogy:**
Imagine a shipping crate holding photo albums:
- $B$ (Batch): The number of separate photo albums in the crate.
- $C$ (Channels): The number of color layers per page (e.g., 1 for grayscale, 3 for Red/Green/Blue).
- $H$ (Height): The number of rows of pixels from top to bottom.
- $W$ (Width): The number of columns of pixels from left to right.

**3. Tiny example worked by hand:**
For Fashion-MNIST, images are grayscale ($C = 1$) and $28 \times 28$ pixels ($H = 28, W = 28$).
If we load a mini-batch of 64 images, our tensor shape is:
$$(B, C, H, W) = (64, 1, 28, 28)$$
Total pixel values in this batch = $64 \times 1 \times 28 \times 28 = 50,176$.

**4. The math with every symbol explained in words:**
$$X \in \mathbb{R}^{B \times C \times H \times W}$$
- $B$: Batch dimension index $b \in \{0, \dots, B-1\}$ (which sample).
- $C$: Channel dimension index $c \in \{0, \dots, C-1\}$ (which color band).
- $H$: Height coordinate $h \in \{0, \dots, H-1\}$ (vertical row index).
- $W$: Width coordinate $w \in \{0, \dots, W-1\}$ (horizontal column index).

**5. Code explained line by line:**


In [4]:
# 1. Simulate a batch of 8 Fashion-MNIST images (B=8, C=1, H=28, W=28)
batch_images = torch.rand(8, 1, 28, 28)

# 2. Inspect shape tuple
print("Batch shape:", batch_images.shape)
# 3. Inspect individual dimensions
b, c, h, w = batch_images.shape
print(f"Batch size (B): {b} images")
print(f"Channels (C): {c} channel (grayscale)")
print(f"Height (H): {h} pixel rows")
print(f"Width (W): {w} pixel columns")
# 4. Check rank / number of dimensions
print("Number of dimensions (ndim):", batch_images.ndim)


Batch shape: torch.Size([8, 1, 28, 28])
Batch size (B): 8 images
Channels (C): 1 channel (grayscale)
Height (H): 28 pixel rows
Width (W): 28 pixel columns
Number of dimensions (ndim): 4


**What we just saw:**
We simulated a 4D batch tensor and unpacked its 4 dimensions. In PyTorch vision pipelines, the channel dimension always precedes height and width ($B, C, H, W$), unlike standard NumPy/OpenCV image arrays which often store $(H, W, C)$.

**6. One common beginner mistake:**
Mixing up Height and Width or confusing $(B, C, H, W)$ with $(B, H, W, C)$. Passing an image batch shaped $(64, 28, 28, 1)$ into `torch.nn.Conv2d` will fail with an error complaining that input channels do not match!

**7. Quick "check yourself" question:**
*Question:* If you have 16 RGB images that are 64 pixels high and 32 pixels wide, what is the exact shape of this tensor in PyTorch?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Shape is <code>(16, 3, 64, 32)</code>: $B=16$, $C=3$ (RGB), $H=64$, $W=32$.
</details>

**Short Exercise 1.2:**
Create a dummy batch representing 4 color (RGB) images of size $32 \times 32$. Print its shape and verify that the channel dimension is 3.


In [5]:
# Exercise 1.2: Create a dummy RGB image batch
# 1. Create tensor of shape (B=4, C=3, H=32, W=32)
rgb_batch = torch.zeros(4, 3, 32, 32)
# 2. Verify shape and channel count
print("RGB batch shape:", rgb_batch.shape)
assert rgb_batch.shape[1] == 3, "Channel dimension must be 3 for RGB!"


RGB batch shape: torch.Size([4, 3, 32, 32])


### 1.3 Data Types (dtypes)
**1. The idea in ONE plain sentence:**
Data type (`dtype`) dictates the binary representation and memory precision used to store each individual number.

**2. Real-life analogy:**
Think of choosing between lined index cards: an integer card only accepts whole numbers (like counting items: 0, 1, 2), while a floating-point card has decimal precision allowing fine fractions (like 0.8371).

**3. Tiny example worked by hand:**
- `torch.float32`: 32 bits (4 bytes) per number. Standard default for neural network weights and activations.
- `torch.int64` (or `torch.long`): 64 bits (8 bytes) per whole number. Required for classification class indices in `nn.CrossEntropyLoss`.
- `torch.float16`: 16 bits (2 bytes). Used in mixed-precision GPU acceleration to halve VRAM usage.

**4. The math with every symbol explained in words:**
Memory footprint:
$$\text{Memory (bytes)} = N \times S$$
- $N$: Total number of elements.
- $S$: Byte size per element ($4$ bytes for float32, $8$ bytes for int64, $2$ bytes for float16).
For 1,000,000 float32 numbers: $1{,}000{,}000 \times 4 = 4{,}000{,}000\text{ bytes} \approx 3.81\text{ MB}$.

**5. Code explained line by line:**


In [6]:
# 1. Create a floating point tensor (default is float32)
f_tensor = torch.tensor([1.5, 2.7, 3.2])
# 2. Inspect default dtype
print("Float tensor dtype:", f_tensor.dtype)

# 3. Create an integer tensor (default from Python ints is int64 / long)
i_tensor = torch.tensor([0, 1, 9])
print("Integer tensor dtype:", i_tensor.dtype)

# 4. Explicitly convert float tensor to 16-bit half precision
half_tensor = f_tensor.to(torch.float16)
print("Half precision dtype:", half_tensor.dtype)

# 5. Explicitly convert integer labels to long (standard for cross entropy targets)
target_tensor = i_tensor.long()
print("Target labels dtype:", target_tensor.dtype)


Float tensor dtype: torch.float32
Integer tensor dtype: torch.int64
Half precision dtype: torch.float16
Target labels dtype: torch.int64


**What we just saw:**
We checked default types and used `.to(...)` and `.long()` to convert between datatypes.

**6. One common beginner mistake:**
Passing float labels (e.g. `torch.tensor([0.0, 1.0])`) to `nn.CrossEntropyLoss`. PyTorch will crash with `RuntimeError: Expected floating point type for input with exception of long for target`. Classification targets must always be `torch.int64` (`torch.long`).

**7. Quick "check yourself" question:**
*Question:* How much memory in megabytes does a tensor of $10^7$ numbers consume when stored in `float32` vs `float16`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> In float32 (4 bytes/element), $10^7 \times 4 = 40\text{ MB}$. In float16 (2 bytes/element), $10^7 \times 2 = 20\text{ MB}$ (50% memory savings!).
</details>

**Short Exercise 1.3:**
Create an integer tensor `[0, 2, 4]` and cast it to `torch.float32`. Print its dtype before and after.


In [7]:
# Exercise 1.3: Convert integer tensor to float32
# 1. Create integer tensor
ex_int = torch.tensor([0, 2, 4])
print("Before cast:", ex_int.dtype)
# 2. Cast to float32 using .float()
ex_float = ex_int.float()
print("After cast:", ex_float.dtype)
assert ex_float.dtype == torch.float32, "Expected torch.float32!"


Before cast: torch.int64
After cast: torch.float32


### 1.4 Indexing and Slicing
**1. The idea in ONE plain sentence:**
Indexing and slicing allow you to access single elements or contiguous sub-regions of a tensor along any dimension.

**2. Real-life analogy:**
Think of street addresses: city (dimension 0), street (dimension 1), house number (dimension 2). Specifying `[2, :, 0]` means: pick city 2, visit every street (`:`), and knock on house number 0.

**3. Tiny example worked by hand:**
Given a $3 \times 3$ grid:
$$\begin{bmatrix} 10 & 20 & 30 \\ 40 & 50 & 60 \\ 70 & 80 & 90 \end{bmatrix}$$
- Element at row 1, col 2: `[1, 2]` $\rightarrow 60$.
- First two rows, all columns: `[0:2, :]` $\rightarrow \begin{bmatrix} 10 & 20 & 30 \\ 40 & 50 & 60 \end{bmatrix}$.
- Column 0 across all rows: `[:, 0]` $\rightarrow [10, 40, 70]$.

**4. The math with every symbol explained in words:**
Sub-slice $T[s_0:e_0, s_1:e_1]$ selects indices:
- $i_0 \in [s_0, e_0)$ along axis 0.
- $i_1 \in [s_1, e_1)$ along axis 1.
Notice that the end index is always non-inclusive (half-open interval $[s, e)$).

**5. Code explained line by line:**


In [8]:
# 1. Create a 3x3 matrix with concrete values
mat = torch.tensor([[10.0, 20.0, 30.0],
                    [40.0, 50.0, 60.0],
                    [70.0, 80.0, 90.0]])

# 2. Extract single item at row 1, column 2
item = mat[1, 2]
print("Single item at mat[1, 2]:", item.item())

# 3. Extract the entire first column (all rows, column 0)
col_0 = mat[:, 0]
print("First column shape:", col_0.shape, "| values:", col_0)

# 4. Extract top 2x2 submatrix
sub_mat = mat[0:2, 0:2]
print("Top 2x2 submatrix shape:", sub_mat.shape)
print(sub_mat)


Single item at mat[1, 2]: 60.0
First column shape: torch.Size([3]) | values: tensor([10., 40., 70.])
Top 2x2 submatrix shape: torch.Size([2, 2])
tensor([[10., 20.],
        [40., 50.]])


**What we just saw:**
We indexed individual elements and sliced submatrices. Note `.item()` extracts a 0D tensor as a standard Python float.

**6. One common beginner mistake:**
Forgetting that basic slicing creates a *view* that shares underlying memory with the original tensor. If you modify `sub_mat[0, 0] = 999.0`, the original `mat[0, 0]` also turns into 999.0! If you need an isolated copy, call `.clone()`.

**7. Quick "check yourself" question:**
*Question:* Given a batch of images `x` with shape `(64, 1, 28, 28)`, what slice expression extracts only the very first image while keeping all 4 dimensions intact?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> <code>x[0:1, :, :, :]</code> (or <code>x[0:1]</code>), which returns shape <code>(1, 1, 28, 28)</code>. (Using <code>x[0]</code> drops the batch dimension, returning <code>(1, 28, 28)</code>).
</details>

**Short Exercise 1.4:**
Extract the last row of `mat` using negative indexing (`mat[-1, :]`) and verify its shape is `(3,)`.


In [9]:
# Exercise 1.4: Extract the last row using negative indexing
# 1. Slice last row
last_row = mat[-1, :]
# 2. Print values and shape
print("Last row values:", last_row)
print("Last row shape:", last_row.shape)
assert last_row.shape == torch.Size([3]), "Shape must be (3,)!"


Last row values: tensor([70., 80., 90.])
Last row shape: torch.Size([3])


### 1.5 Broadcasting
**1. The idea in ONE plain sentence:**
Broadcasting automatically duplicates dimensions of length 1 during arithmetic so tensors of different shapes can be combined without copying memory.

**2. Real-life analogy:**
Imagine a teacher applying a curved grade adjustment: the teacher writes `+5 points` once on the blackboard (a scalar or single row), and it automatically gets added to every single student's exam score on the roster.

**3. Tiny example worked by hand:**
Add a row matrix $A$ of shape $(1, 3)$ and a column matrix $B$ of shape $(2, 1)$:
$$A = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}, \quad B = \begin{bmatrix} 10 \\ 20 \end{bmatrix}$$
Broadcasting expands $A$ along rows and $B$ along columns:
$$A_{\text{expanded}} = \begin{bmatrix} 1 & 2 & 3 \\ 1 & 2 & 3 \end{bmatrix}, \quad B_{\text{expanded}} = \begin{bmatrix} 10 & 10 & 10 \\ 20 & 20 & 20 \end{bmatrix}$$
$$A + B = \begin{bmatrix} 1+10 & 2+10 & 3+10 \\ 1+20 & 2+20 & 3+20 \end{bmatrix} = \begin{bmatrix} 11 & 12 & 13 \\ 21 & 22 & 23 \end{bmatrix} \quad \text{shape } (2, 3)$$

**4. The math with every symbol explained in words:**
Two shapes $(a_1, a_2, \dots)$ and $(b_1, b_2, \dots)$ are compatible for broadcasting if, comparing dimensions from right to left:
- They are equal ($a_i = b_i$), OR
- One of them is $1$ ($a_i = 1$ or $b_i = 1$), OR
- One dimension does not exist (it is implicitly treated as $1$).

**5. Code explained line by line:**


In [10]:
# 1. Create row tensor of shape (1, 3)
row_vec = torch.tensor([[1.0, 2.0, 3.0]])
# 2. Create column tensor of shape (2, 1)
col_vec = torch.tensor([[10.0], [20.0]])

# 3. Add them together with automatic broadcasting
broadcasted_sum = row_vec + col_vec

# 4. Print input shapes and output shape/result
print("row_vec shape:", row_vec.shape)
print("col_vec shape:", col_vec.shape)
print("broadcasted_sum shape:", broadcasted_sum.shape)
print("Resulting matrix:")
print(broadcasted_sum)


row_vec shape: torch.Size([1, 3])
col_vec shape: torch.Size([2, 1])
broadcasted_sum shape: torch.Size([2, 3])
Resulting matrix:
tensor([[11., 12., 13.],
        [21., 22., 23.]])


**What we just saw:**
PyTorch automatically aligned the shapes $(1, 3)$ and $(2, 1)$ to produce $(2, 3)$ matching our hand calculation.

**6. One common beginner mistake:**
Accidentally broadcasting a 1D vector of predictions of shape `(N,)` with targets of shape `(N, 1)`. When you compute `loss = pred - target`, PyTorch does NOT throw an error; instead, it broadcasts them into a giant $(N, N)$ matrix! If $N=10{,}000$, it creates a $100{,}000{,}000$-element matrix, quietly exhausting memory. Always ensure shapes match before computing loss!

**7. Quick "check yourself" question:**
*Question:* Can a tensor of shape `(5, 1, 4)` and a tensor of shape `(1, 3, 4)` be added via broadcasting? What is the output shape?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Yes! Comparing dimensions right to left: $4=4$ (match), $1$ and $3$ (compatible $\rightarrow 3$), $5$ and $1$ (compatible $\rightarrow 5$). Output shape is <code>(5, 3, 4)</code>.
</details>

**Short Exercise 1.5:**
Given a batch of image features `features = torch.ones(4, 10)` and a bias vector `bias = torch.tensor([0.5] * 10)`, add them and verify the output shape is `(4, 10)`.


In [11]:
# Exercise 1.5: Broadcast bias vector across feature batch
# 1. Batch of 4 samples with 10 features each
features = torch.ones(4, 10)
# 2. Bias vector with 10 features (shape (10,))
bias = torch.full((10,), 0.5)
# 3. Add bias to features
features_with_bias = features + bias
# 4. Confirm shape and values
print("features_with_bias shape:", features_with_bias.shape)
print("Sample row:", features_with_bias[0])
assert features_with_bias.shape == torch.Size([4, 10]), "Shape must remain (4, 10)!"


features_with_bias shape: torch.Size([4, 10])
Sample row: tensor([1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000])


### 1.6 Reshape vs View (and Memory Contiguity)
**1. The idea in ONE plain sentence:**
Changing the shape of a tensor can either create a zero-cost "view" looking at existing memory or create a new memory copy if the elements are not sequentially laid out in RAM.

**2. Real-life analogy:**
Imagine 12 toy blocks sitting in a straight row on a table. Calling `.view(3, 4)` is like simply saying "let's count every 4 blocks as a new group." You didn't move a single block. But if someone scrambles the blocks across the room (transposing), you must physically pick them up and re-align them in order before grouping them again.

**3. Tiny example worked by hand:**
Take a $2 \times 3$ matrix:
$$\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}$$
In physical memory, the numbers sit in linear row-major order: `[1, 2, 3, 4, 5, 6]`.
- Flattening to shape `(6,)` via `.view(6)` just changes the shape label.
- Transposing to shape `(3, 2)` gives:
  $$\begin{bmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{bmatrix}$$
  The elements in memory are still physically stored as `[1, 2, 3, 4, 5, 6]`, but the stride changed. Calling `.view()` directly on this transposed tensor fails because memory is non-contiguous!

**4. The math with every symbol explained in words:**
- **Stride:** For a tensor $T$, `stride[i]` is the number of memory addresses skipped to move by $1$ along dimension $i$.
- A tensor is **contiguous** if:
  $$\text{stride}[k] = 1 \quad \text{and} \quad \text{stride}[i] = \text{stride}[i+1] \times \text{size}[i+1]$$
- `.reshape()` checks: if contiguous, it calls `.view()`; if non-contiguous, it calls `.contiguous().view()` automatically.

**5. Code explained line by line:**


In [12]:
# 1. Create a contiguous 2x3 tensor
t_orig = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("t_orig is_contiguous:", t_orig.is_contiguous())
print("t_orig strides:", t_orig.stride())

# 2. Reshape to (6,) using .view() (zero memory copy)
t_flat_view = t_orig.view(6)
print("t_flat_view shape:", t_flat_view.shape)

# 3. Transpose matrix (swaps rows and columns)
t_transposed = t_orig.t()
print("t_transposed shape:", t_transposed.shape)
print("t_transposed is_contiguous:", t_transposed.is_contiguous())

# 4. Demonstrate why .reshape() is safer than .view()
# Calling .view(6) on t_transposed would throw RuntimeError!
t_safe_flat = t_transposed.reshape(6)
print("t_safe_flat succeeds! Shape:", t_safe_flat.shape, "| Values:", t_safe_flat)


t_orig is_contiguous: True
t_orig strides: (3, 1)
t_flat_view shape: torch.Size([6])
t_transposed shape: torch.Size([3, 2])
t_transposed is_contiguous: False
t_safe_flat succeeds! Shape: torch.Size([6]) | Values: tensor([1, 4, 2, 5, 3, 6])


**What we just saw:**
Transposing changed the strides without moving memory, rendering `t_transposed` non-contiguous (`is_contiguous() == False`). Calling `.reshape()` handled this cleanly by re-arranging memory under the hood.

**6. One common beginner mistake:**
Calling `.view(...)` after transpose or permutation operations without `.contiguous()`. It crashes with: `RuntimeError: view size is not compatible with input tensor's size and stride (tensor is not contiguous)`. Using `.reshape(...)` or `.contiguous().view(...)` fixes this error every time.

**7. Quick "check yourself" question:**
*Question:* When should you strictly prefer `.view()` over `.reshape()`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Use <code>.view()</code> when you want an absolute architectural guarantee that no memory copy is created (for strict zero-copy efficiency). If a copy is ever required, <code>.view()</code> intentionally errors out so you know immediately. Otherwise, <code>.reshape()</code> is the safer and more flexible choice.
</details>

**Short Exercise 1.6:**
Flatten a batch of dummy images of shape `(16, 1, 28, 28)` into a 2D matrix of shape `(16, 784)` using `.reshape()`.


In [13]:
# Exercise 1.6: Flatten image batch for linear layers
# 1. Create dummy batch (B=16, C=1, H=28, W=28)
dummy_batch = torch.rand(16, 1, 28, 28)
# 2. Flatten spatial dimensions into 784 features per sample
flattened_batch = dummy_batch.reshape(16, -1)
# 3. Inspect shape (-1 automatically computed 1*28*28 = 784)
print("Flattened batch shape:", flattened_batch.shape)
assert flattened_batch.shape == torch.Size([16, 784]), "Shape must be (16, 784)!"


Flattened batch shape: torch.Size([16, 784])


### 1.7 Moving Between CPU and GPU (Automatic Device Selection)
**1. The idea in ONE plain sentence:**
Tensors live in either system RAM (`cpu`) or graphics card memory (`cuda`), and operations only work between tensors living on the exact same device.

**2. Real-life analogy:**
Think of a master chef in the kitchen (GPU) and a bookkeeper in the front office (CPU). You cannot mix flour and eggs if the flour is on the kitchen counter and the eggs are locked inside the bookkeeper's desk. You must bring all ingredients into the kitchen first.

**3. Tiny example worked by hand:**
- CPU tensor $A$ lives in system RAM.
- GPU tensor $B$ lives in dedicated GDDR VRAM.
- $A + B$ $\rightarrow$ PyTorch raises a device mismatch error.
- Move $A$ to GPU: $A_{\text{gpu}} = A\text{.to}(\text{"cuda"})$.
- Now $A_{\text{gpu}} + B$ executes in parallel across thousands of GPU cores.

**4. The math / Hardware architecture:**
Data transfer occurs across the PCIe bus:
$$\text{Transfer Time} = \frac{\text{Tensor Bytes}}{\text{PCIe Bandwidth}}$$
While PCIe transfer has a small overhead, once tensors reside in GPU VRAM, matrix operations run 10x-100x faster than on CPU.

**5. Code explained line by line:**


In [14]:
# 1. Create a tensor on CPU
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])
print("Initial device:", cpu_tensor.device)

# 2. Move tensor to our globally selected device (cuda or cpu)
dev_tensor = cpu_tensor.to(device)
print("Device after .to(device):", dev_tensor.device)

# 3. Move back to CPU explicitly
back_to_cpu = dev_tensor.cpu()
print("Device after .cpu():", back_to_cpu.device)

# 4. Check whether tensor is currently on CUDA
print("Is tensor on CUDA?:", dev_tensor.is_cuda)


Initial device: cpu
Device after .to(device): cuda:0
Device after .cpu(): cpu
Is tensor on CUDA?: True


**What we just saw:**
We moved tensors between CPU and GPU with `.to(device)`. The `.is_cuda` attribute confirms whether a tensor lives in graphics VRAM.

**6. One common beginner mistake:**
Calling `model.to(device)` but forgetting to call `images = images.to(device)` in the training loop. This produces the infamous error: `RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!`.

**7. Quick "check yourself" question:**
*Question:* Does `tensor.to(device)` modify `tensor` in place, or does it return a new tensor?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> For tensors, <code>.to(device)</code> returns a <b>new copy</b> on the target device! You must write <code>tensor = tensor.to(device)</code>. (Note: for <code>nn.Module</code> models, <code>model.to(device)</code> modifies the model in place, which makes this a very frequent beginner trap!).
</details>

**Short Exercise 1.7:**
Write a small helper function `ensure_device(tensor, target_device)` that checks if a tensor is already on `target_device`, and moves it only if necessary.


In [15]:
# Exercise 1.7: Device transfer helper function
def ensure_device(tensor: torch.Tensor, target_device: torch.device) -> torch.Tensor:
    # 1. Check if tensor already resides on target_device
    if tensor.device != target_device:
        # 2. Move tensor to target device
        return tensor.to(target_device)
    # 3. Return original tensor without copying
    return tensor

# Test helper function
sample_t = torch.tensor([42.0])
verified_t = ensure_device(sample_t, device)
print(f"Verified device: {verified_t.device}")
assert verified_t.device == device, "Device mismatch!"


Verified device: cuda:0


### 1.8 NumPy Interoperability
**1. The idea in ONE plain sentence:**
PyTorch CPU tensors and NumPy arrays can convert back and forth without copying memory because they share the exact same underlying RAM buffer.

**2. Real-life analogy:**
Putting two different name tags on the same person: one tag says "PyTorch Tensor" and the other says "NumPy Array". If that person changes their shirt, both name tags show the change because there is only one physical person.

**3. Tiny example worked by hand:**
- Create NumPy array: `a = np.array([1.0, 2.0])`.
- Convert to PyTorch: `t = torch.from_numpy(a)`.
- Change `a[0] = 99.0`.
- Look at `t[0]`: it is automatically `99.0` because no memory was duplicated!

**4. The math / Memory pointer:**
Both the PyTorch tensor descriptor and the NumPy array descriptor store pointers pointing to the exact same physical memory address in host RAM:
$$\text{address}(t.\text{data\_ptr}()) == \text{address}(a.__\text{array\_interface__}['\text{data}'][0])$$

**5. Code explained line by line:**


In [16]:
# 1. Create a NumPy array
np_arr = np.array([10.0, 20.0, 30.0], dtype=np.float32)

# 2. Convert to PyTorch tensor (shares memory!)
t_from_np = torch.from_numpy(np_arr)
print("Initial PyTorch tensor from NumPy:", t_from_np)

# 3. Modify NumPy array in place
np_arr[0] = 999.0

# 4. Observe that the PyTorch tensor changed simultaneously!
print("PyTorch tensor after NumPy modification:", t_from_np)

# 5. Convert PyTorch tensor back to NumPy
np_back = t_from_np.numpy()
print("Converted back to NumPy type:", type(np_back), "| values:", np_back)


Initial PyTorch tensor from NumPy: tensor([10., 20., 30.])
PyTorch tensor after NumPy modification: tensor([999.,  20.,  30.])
Converted back to NumPy type: <class 'numpy.ndarray'> | values: [999.  20.  30.]


**What we just saw:**
Modifying the NumPy array immediately updated the PyTorch tensor because they share memory. Calling `.numpy()` converts back to NumPy instantly.

**6. One common beginner mistake:**
Calling `.numpy()` directly on a GPU tensor. PyTorch raises: `TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() first`. NumPy cannot access GPU VRAM directly; you must always call `tensor.cpu().numpy()`.

**7. Quick "check yourself" question:**
*Question:* How do you convert a PyTorch tensor to NumPy WITHOUT sharing memory so future edits don't affect each other?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Call <code>tensor.clone().numpy()</code> (or <code>np_arr.copy()</code>). The <code>.clone()</code> method allocates a brand new memory buffer.
</details>

**Short Exercise 1.8:**
Convert a GPU tensor to a NumPy array safely using `.detach().cpu().numpy()`.


In [17]:
# Exercise 1.8: Safe GPU-to-NumPy conversion
# 1. Create a tensor on our device
gpu_sample = torch.tensor([1.23, 4.56]).to(device)
# 2. Safely move to CPU and convert to NumPy
np_result = gpu_sample.detach().cpu().numpy()
# 3. Inspect result
print("NumPy result from device tensor:", np_result, "| Type:", type(np_result))
assert isinstance(np_result, np.ndarray), "Expected NumPy ndarray!"


NumPy result from device tensor: [1.23 4.56] | Type: <class 'numpy.ndarray'>


### Step 1 Recap
- **Recap in 3 bullets:**
  1. Tensors are multi-dimensional grids of numbers characterized by their shape (e.g. $B, C, H, W$ in vision) and data type (`float32` for weights, `long` for labels).
  2. Broadcasting expands length-1 dimensions automatically, while `.view()`/`.reshape()` re-indexes memory layout; use `.reshape()` to avoid contiguity pitfalls.
  3. Tensors must reside on the same device (`cpu` or `cuda`) to interact; PyTorch CPU tensors share memory directly with NumPy arrays.

- **If you remember only one thing:**
  *Always verify tensor shapes and devices at every boundary: mismatched dimensions or mismatched devices cause 90% of beginner PyTorch bugs.*
